# CMB residuals of two models w.r.t. a mock

Residuals of the lensed **TT, TE, EE** and the lensing **dd** spectra of fitted models, relative to a mock,
$C_\ell^{\rm model} - C_\ell^{\rm mock}$ (mock noise removed), for any MontePython runs in the `with_*` pipelines.

- **`MODE = "bestfit"`**: one curve per model, the minimum $-\ln L$ sample of its chains.
- **`MODE = "posterior"`**: `N_SAMPLES` posterior draws (weighted, after burn-in) → median and 68% / 95% bands
  of the residual at each $\ell$ (or $\ell$-bin); best fit overlaid as a dashed line if `SHOW_BESTFIT`.

Spectra are recomputed with the pipeline each run was sampled with (the `classy.py` under its `log.param` `data.path['cosmo']`,
remapped to this machine), with the run's fixed parameters (`l_switch`, `l_c`, …) and `data.cosmo_arguments`.
Noise, $f_{\rm sky}$ and $\ell$ range come from the run's likelihood settings in `log.param`.

Residual kinds (`RESIDUAL`):
- `"sigma"`: $\Delta C_\ell/\sigma_\ell$, $\sigma_\ell$ = Gaussian error of the mock spectrum (cosmic variance + noise, $f_{\rm sky}$; diagonal per spectrum,
  i.e. ignoring the TT–TE–EE correlations of the likelihood). Binned: $\sum w\,\Delta C/\sqrt{\sum w}$, $w=1/\sigma_\ell^2$ = bandpower deviation in bandpower σ.
- `"frac"`: $\Delta C_\ell / C_\ell^{\rm mock}$ (TE: $/\sqrt{C^{TT}_\ell C^{EE}_\ell}$). Binned: inverse-variance weighted mean.
- `"dl"`: $\Delta D_\ell = \ell(\ell+1)\Delta C_\ell/2\pi$ in μK² (dd: $[\ell(\ell+1)]^2\Delta C^{\phi\phi}_\ell/2\pi$). Binned: inverse-variance weighted mean.

The compute cell (2) only depends on the runs / mode / samples; the plot cell (3) can be re-run with another `RESIDUAL`, `BIN_WIDTH`, `SPECTRA`.
The kernel needs the pipeline's dependencies (for `with_cp_lcdm`: the `cp_lcdm` / `cosmopower` env).

In [ ]:
from pathlib import Path
import re
import numpy as np
import matplotlib.pyplot as plt
import residuals as R          # cmb_residuals/residuals.py

# ---- input ------------------------------------------------------------------
CH = "with_cp_lcdm/chains"     # paths are relative to the omega_m/ root
MOCK = "with_maxiclass/model/ede2ax_pdh0_mean.dat"
MODELS = {                     # label: MP output folder  (2 models; more also work)
    r"$\Lambda$CDM (one_bin)":       f"{CH}/model_ede2ax_pdh0_mean_from_maxiclass__param_one_bin",
    r"$\omega_m(\ell)$ poly":        f"{CH}/model_ede2ax_pdh0_mean_from_maxiclass__param_poly",
}
ROOTS = {}                     # optional {label: chain root, e.g. "2026-10-22_100000_"}; default: newest in folder
COLORS = ["C0", "C3", "C2", "C1"]

MODE = "posterior"             # "bestfit" or "posterior"
N_SAMPLES = 300                # posterior draws per model (with_cp_lcdm: ~0.03 s each)
BURN_IN = 0.3
SEED = 0
SHOW_BESTFIT = True            # posterior mode: overlay the best fit (dashed)

# ---- plot -------------------------------------------------------------------
RESIDUAL = "sigma"             # "sigma", "frac" or "dl"
BIN_WIDTH = 30                 # l-bin width (1 or None: per-l), or an array of edges,
                               # e.g. np.geomspace(2, 2509, 60) with XSCALE = "log"
SPECTRA = ("tt", "te", "ee", "dd")
XSCALE = "linear"              # or "log"
SHOW_BIN_EDGES = True          # dotted lines at the fixed l_switch of binned-omega_m runs
SAVE = True                    # PNG in cmb_residuals/plots/

In [ ]:
runs = {lab: R.Run(f, root=ROOTS.get(lab), burn_in=BURN_IN) for lab, f in MODELS.items()}
first = next(iter(runs.values()))
mock = R.Mock(MOCK, first)
for lab, run in runs.items():
    for k in ("l_min", "l_max", "f_sky", "noise_file"):
        assert run.lik[k] == first.lik[k], f"{lab}: likelihood setting {k} differs from the first model"
    if run.fiducial_file is not None and run.fiducial_file.resolve() != mock.path.resolve():
        print(f"NOTE {lab}: sampled on {run.fiducial_file.relative_to(R.ROOT)}, plotted against {MOCK}")
print(mock.header[0])

def stack(list_of_cl):
    return {k: np.array([c[k] for c in list_of_cl]) for k in R.SPECTRA}

best = {lab: R.spectra(run, run.bestfit()) for lab, run in runs.items()}
post = {}
if MODE == "posterior":
    for lab, run in runs.items():
        post[lab] = stack([R.spectra(run, p) for p in run.draws(N_SAMPLES, seed=SEED)])
        print(f"{lab}: {N_SAMPLES} draws done")

In [ ]:
edges = R.bins(mock, BIN_WIDTH)
btag = ("1" if edges is None else str(BIN_WIDTH)) if np.isscalar(BIN_WIDTH) or BIN_WIDTH is None else "custom"
fig, axes = plt.subplots(len(SPECTRA), 1, figsize=(9, 2.3 * len(SPECTRA) + 0.6), sharex=True, squeeze=False)
axes = axes[:, 0]
for ax, spec in zip(axes, SPECTRA):
    ax.axhline(0, color="k", lw=0.8)
    for (lab, run), col in zip(runs.items(), COLORS):
        lb, rb = R.residual(best[lab], spec, mock, RESIDUAL, edges)
        if MODE == "posterior":
            r = np.array([R.residual({spec: c}, spec, mock, RESIDUAL, edges)[1] for c in post[lab][spec]])
            q = np.percentile(r, [2.5, 16, 50, 84, 97.5], axis=0)
            ax.fill_between(lb, q[0], q[4], color=col, alpha=0.15, lw=0)
            ax.fill_between(lb, q[1], q[3], color=col, alpha=0.35, lw=0)
            ax.plot(lb, q[2], color=col, lw=1.3, label=f"{lab} (median, 68/95%)")
            if SHOW_BESTFIT:
                ax.plot(lb, rb, color=col, lw=1.0, ls="--", label=f"{lab} best fit")
        else:
            ax.plot(lb, rb, color=col, lw=1.3, label=f"{lab} best fit")
        if SHOW_BIN_EDGES:
            for e in run.l_switch:
                ax.axvline(e, color=col, ls=":", lw=0.7)
    ax.set_ylabel(R.ylabel(RESIDUAL, spec))
axes[0].legend(fontsize=8, ncol=2, loc="lower left", bbox_to_anchor=(0, 1.02), frameon=False)
axes[-1].set_xlabel(r"multipole $\ell$" + ("" if edges is None else
                    f"  (bins of {BIN_WIDTH})" if btag != "custom" else f"  ({len(edges) - 1} bins)"))
axes[-1].set_xscale(XSCALE)
axes[-1].set_xlim(mock.ell[0], mock.ell[-1])
fig.suptitle(f"residuals w.r.t. mock {mock.name}  ({MODE}"
             + (f", {N_SAMPLES} draws" if MODE == "posterior" else "") + ")", fontsize=10)
fig.tight_layout()
if SAVE:
    out = Path(R.__file__).parent / "plots"
    out.mkdir(exist_ok=True)
    tag = "_vs_".join(re.sub(r".*__param_", "", Path(f).name) for f in MODELS.values())
    fig.savefig(out / f"{mock.name}__{tag}__{MODE}_{RESIDUAL}_b{btag}.png", dpi=150)
plt.show()